# Awinda: IMU IK → ID vs OpenSim ID

Compares checkpoint inference from IMU IK against OpenSim inverse dynamics for **awinda** (no-exo) trials.

- **Data**: `/media/metamobility3/Samsung_T52/Results/processed`
- **Sync**: first right-hip-flexion angle peak (>15°)
- **Filters**: match training config (`zero_phase` 6 Hz angle/output, 15 Hz velocity)
- **Metrics**: RMSE and R² (per joint + overall)

Use kernel `jinwoo-addbiomech` (needs PyTorch).

In [2]:
import sys
import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from scipy.signal import butter, sosfilt, sosfiltfilt, find_peaks
import ipywidgets as widgets
from IPython.display import display

warnings.filterwarnings('ignore', message='.*NumPy.*')

PROJECT_ROOT = Path('/home/metamobility3/Jinwoo/os_kinetics').resolve()
PROCESSED_ROOT = Path('/media/metamobility3/Samsung_T52/Results/processed')
IMU_IK_ROOT = Path('/home/metamobility3/Jinwoo/mt_processed')
IMU_IK_METHOD = 'VQF'
CHECKPOINT = PROJECT_ROOT / 'runs/0512_ik_id_all_zero_in_zero_out/best_model.pt'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

SUBJECT_MASS_KG = {
    'AB01_Jinwoo': 88.0, 'AB02_Oscar': 71.1, 'AB03_Ilseung': 84.4,
    'AB04_Changseob': 74.0, 'AB05_Maria': 55.0, 'AB06_Jimin': 82.6,
    'AB07_Amy': 51.3, 'AB08_Seokhyun': 71.9,
}

CHANNELS = [
    'hip_flexion_r', 'knee_angle_r', 'ankle_angle_r',
    'hip_flexion_l', 'knee_angle_l', 'ankle_angle_l',
]
ID_COLS = [f'{c}_moment' for c in CHANNELS]
DEFAULT_FS_HZ = 100.0
IK_ALIGN_MAX_LAG_SEC = 30.0
PEAK_THRESHOLD_DEG = 15.0

sys.path.insert(0, str(PROJECT_ROOT))
from dataset import IK_DOF_NAMES
from model import TCN

print(f'Checkpoint: {CHECKPOINT}')
print(f'Processed root: {PROCESSED_ROOT}')
print(f'Device: {DEVICE}')

Checkpoint: /home/metamobility3/Jinwoo/os_kinetics/runs/0512_ik_id_all_zero_in_zero_out/best_model.pt
Processed root: /media/metamobility3/Samsung_T52/Results/processed
Device: cuda


In [3]:
def parse_opensim_table(path: Path) -> pd.DataFrame:
    with open(path) as f:
        header_end = next(i for i, line in enumerate(f) if line.strip().lower() == 'endheader')
    return pd.read_csv(path, sep=r'\s+', skiprows=header_end + 1).set_index('time')


def butter_lpf(x, fs_hz, cutoff_hz, order, mode='zero_phase'):
    x = np.asarray(x, dtype=np.float64).reshape(-1)
    nyq = 0.5 * fs_hz
    if cutoff_hz <= 0 or cutoff_hz >= nyq or len(x) < 4:
        return x.copy()
    sos = butter(order, cutoff_hz / nyq, btype='low', output='sos')
    return sosfiltfilt(sos, x) if mode == 'zero_phase' else sosfilt(sos, x)


def lpf_mc(X, fs_hz, cutoff_hz, order, mode='zero_phase'):
    return np.column_stack([butter_lpf(X[:, c], fs_hz, cutoff_hz, order, mode) for c in range(X.shape[1])])


def filename_to_condition(stem: str) -> str:
    speed, cond = stem.split('_', 1)
    return f'{cond.upper()}_{speed}'


def _first_peak_idx(sig, fs_hz, threshold_rad, start_offset_s=0.25):
    x = np.asarray(sig, dtype=np.float64).copy()
    valid = np.isfinite(x)
    if valid.sum() < 20:
        raise RuntimeError('Too few finite samples for peak detection')
    x[~valid] = float(np.nanmedian(x[valid]))
    start_idx = min(len(x) - 1, max(0, int(round(start_offset_s * fs_hz))))
    span = float(np.nanmax(x) - np.nanmin(x))
    peaks, _ = find_peaks(x, prominence=max(np.deg2rad(3.0), 0.10 * span), distance=max(1, int(round(0.3 * fs_hz))))
    peaks = peaks[(peaks >= start_idx) & (x[peaks] >= threshold_rad)]
    if len(peaks):
        return int(peaks[0])
    crossing = np.where(x[start_idx:] >= threshold_rad)[0]
    if len(crossing):
        return int(start_idx + crossing[0])
    raise RuntimeError('No right hip flexion peak found')


def estimate_lag_samples(imu_pos, mocap_pos, fs_hz, max_lag_samples, threshold_deg=15.0):
    hip_idx = IK_DOF_NAMES.index('hip_flexion_r')
    thr = np.deg2rad(threshold_deg)
    imu_idx = _first_peak_idx(imu_pos[:, hip_idx], fs_hz, thr)
    mocap_idx = _first_peak_idx(mocap_pos[:, hip_idx], fs_hz, thr)
    lag = int(imu_idx - mocap_idx)
    clipped = False
    if lag > max_lag_samples:
        lag, clipped = max_lag_samples, True
    elif lag < -max_lag_samples:
        lag, clipped = -max_lag_samples, True
    return lag, {'imu_peak_idx': imu_idx, 'mocap_peak_idx': mocap_idx, 'lag_clipped': clipped}


def build_mocap_ik_rad(ik_df: pd.DataFrame) -> np.ndarray:
    pos_deg = np.full((len(ik_df), len(IK_DOF_NAMES)), np.nan)
    for j, name in enumerate(IK_DOF_NAMES):
        if name in ik_df.columns:
            pos_deg[:, j] = ik_df[name].to_numpy(dtype=np.float64)
    return np.deg2rad(pos_deg)


def build_model_input_from_pkl(imu_dict: dict) -> np.ndarray:
    n = len(next(iter(imu_dict.values())))
    pos_deg = np.zeros((n, len(IK_DOF_NAMES)), dtype=np.float64)
    key_map = {
        'hip_flexion_r': 'hip_flexion_r', 'knee_angle_r': 'knee_flexion_r', 'ankle_angle_r': 'ankle_flexion_r',
        'hip_flexion_l': 'hip_flexion_l', 'knee_angle_l': 'knee_flexion_l', 'ankle_angle_l': 'ankle_flexion_l',
    }
    sign_map = {'knee_angle_r': -1.0, 'knee_angle_l': -1.0}
    for ik_name, pkl_name in key_map.items():
        idx = IK_DOF_NAMES.index(ik_name)
        pos_deg[:, idx] = sign_map.get(ik_name, 1.0) * np.asarray(imu_dict[pkl_name], dtype=np.float64)
    return pos_deg


def rmse_r2(y_pred, y_true):
    m = np.isfinite(y_pred) & np.isfinite(y_true)
    if m.sum() < 2:
        return np.nan, np.nan
    e = y_pred[m] - y_true[m]
    rmse = float(np.sqrt(np.mean(e ** 2)))
    r2 = float(np.corrcoef(y_pred[m], y_true[m])[0, 1] ** 2)
    return rmse, r2

print('Helpers ready.')

Helpers ready.


In [4]:
import json

ckpt = torch.load(str(CHECKPOINT), map_location=DEVICE, weights_only=False)
with open(CHECKPOINT.parent / 'config.json') as f:
    train_cfg = json.load(f)

cfg = ckpt['model_config']
model = TCN(**{k: cfg[k] for k in ['n_input_channels', 'n_output_channels', 'hidden_channels', 'n_blocks', 'kernel_size', 'dropout']})
model.load_state_dict(ckpt['model_state_dict'])
model.to(DEVICE).eval()

WINDOW_SIZE = int(ckpt.get('window_size', 100))
INPUT_INDICES = list(ckpt.get('input_indices', [6, 9, 10, 13, 16, 17]))
H = len(INPUT_INDICES) // 2
INPUT_IDX_R, INPUT_IDX_L = INPUT_INDICES[:H], INPUT_INDICES[H:]

ANGLE_CUTOFF = float(train_cfg.get('lowpass_cutoff_hz', 6.0))
VEL_CUTOFF = float(train_cfg.get('velocity_lowpass_cutoff_hz', 15.0))
OUT_CUTOFF = float(train_cfg.get('lowpass_cutoff_hz', 6.0))
FILTER_ORDER = int(train_cfg.get('lowpass_order', 4))
IN_MODE = str(train_cfg.get('input_lowpass_mode', 'zero_phase'))
OUT_MODE = str(train_cfg.get('output_lowpass_mode', 'zero_phase'))

@torch.no_grad()
def infer_one_side(pos_3, vel_3):
    x = np.concatenate([pos_3, vel_3], axis=1).astype(np.float32)
    n, W, c_out = x.shape[0], WINDOW_SIZE, cfg['n_output_channels']
    pred = np.zeros((n, c_out), dtype=np.float64)
    def _fwd(start):
        xt = torch.from_numpy(np.ascontiguousarray(x[start:start + W].T)).unsqueeze(0).to(DEVICE)
        return model(xt).squeeze(0).detach().cpu().numpy().T
    pred[:W] = _fwd(0)
    for start in range(1, n - W + 1):
        pred[start + W - 1] = _fwd(start)[W - 1]
    return pred.astype(np.float32)


def run_bilateral_inference(pos_full, vel_full):
    pr = infer_one_side(pos_full[:, INPUT_IDX_R], vel_full[:, INPUT_IDX_R])
    pl = infer_one_side(pos_full[:, INPUT_IDX_L], vel_full[:, INPUT_IDX_L])
    return np.concatenate([pr, pl], axis=1)

print(f'window={WINDOW_SIZE}, filters: angle={ANGLE_CUTOFF}Hz/{IN_MODE}, vel={VEL_CUTOFF}Hz/{IN_MODE}, out={OUT_CUTOFF}Hz/{OUT_MODE}')

window=100, filters: angle=6.0Hz/zero_phase, vel=15.0Hz/zero_phase, out=6.0Hz/zero_phase


In [ ]:
records = []
WARNINGS = []

for subj_dir in sorted(IMU_IK_ROOT.glob('AB*')):
    if not subj_dir.is_dir():
        continue
    subject = subj_dir.name
    processed_subj = PROCESSED_ROOT / subject
    if not processed_subj.is_dir():
        WARNINGS.append(f'[WARN] No processed folder for {subject} — skipping all trials')
        continue

    ik_dir = subj_dir / 'ik' / IMU_IK_METHOD
    if not ik_dir.exists():
        ik_dir = subj_dir

    for pkl_path in sorted(ik_dir.glob('*.pkl')):
        cond = filename_to_condition(pkl_path.stem)
        id_path = processed_subj / 'awinda' / 'id' / f'{cond}_id.sto'
        mocap_ik_path = processed_subj / 'awinda' / 'ik' / f'{cond}_ik.mot'
        ok = id_path.exists() and mocap_ik_path.exists()
        if not ok:
            missing = []
            if not id_path.exists():
                missing.append('id')
            if not mocap_ik_path.exists():
                missing.append('ik')
            WARNINGS.append(f'[WARN] {subject}::{cond} missing {missing} in processed/awinda — skipped')
        records.append({
            'subject': subject,
            'condition': cond,
            'pkl_path': pkl_path,
            'id_path': id_path if id_path.exists() else None,
            'mocap_ik_path': mocap_ik_path if mocap_ik_path.exists() else None,
            'mass_kg': SUBJECT_MASS_KG.get(subject, np.nan),
            'ready': ok,
        })

manifest = pd.DataFrame(records)
for w in WARNINGS:
    print(w)
print(f'\nDiscovered {len(manifest)} IMU trials | ready={int(manifest["ready"].sum())} | skipped={int((~manifest["ready"]).sum())}')
display(manifest[['subject', 'condition', 'ready', 'mass_kg']])

[WARN] AB01_Jinwoo::LG_0p8mps missing ['id'] in processed/awinda — skipped
[WARN] AB01_Jinwoo::RA_0p8mps missing ['id'] in processed/awinda — skipped
[WARN] AB01_Jinwoo::RD_0p8mps missing ['id'] in processed/awinda — skipped
[WARN] AB01_Jinwoo::LG_1p2mps missing ['id'] in processed/awinda — skipped
[WARN] AB01_Jinwoo::LG_1p6mps missing ['id'] in processed/awinda — skipped
[WARN] AB04_Changseob::LG_0p8mps missing ['id'] in processed/awinda — skipped
[WARN] AB04_Changseob::RA_0p8mps missing ['id'] in processed/awinda — skipped
[WARN] AB04_Changseob::RD_0p8mps missing ['id'] in processed/awinda — skipped
[WARN] AB04_Changseob::LG_1p2mps missing ['id'] in processed/awinda — skipped
[WARN] AB04_Changseob::LG_1p6mps missing ['id'] in processed/awinda — skipped
[WARN] AB05_Maria::LG_0p8mps missing ['id', 'ik'] in processed/awinda — skipped
[WARN] AB05_Maria::RA_0p8mps missing ['id', 'ik'] in processed/awinda — skipped
[WARN] AB05_Maria::RD_0p8mps missing ['id', 'ik'] in processed/awinda — ski

,subject,condition,ready,mass_kg
0,AB01_Jinwoo,LG_0p8mps,False,88.0
1,AB01_Jinwoo,RA_0p8mps,False,88.0
2,AB01_Jinwoo,RD_0p8mps,False,88.0
3,AB01_Jinwoo,LG_1p2mps,False,88.0
4,AB01_Jinwoo,LG_1p6mps,False,88.0
5,AB02_Oscar,LG_0p8mps,True,71.1
6,AB02_Oscar,RA_0p8mps,True,71.1
7,AB02_Oscar,RD_0p8mps,True,71.1
8,AB02_Oscar,LG_1p2mps,True,71.1
9,AB02_Oscar,LG_1p6mps,True,71.1


In [6]:
TRIAL_DATA = {}
rows = []

for _, row in manifest[manifest['ready']].iterrows():
    subject, cond = row['subject'], row['condition']
    trial_key = f'{subject}::{cond}'
    try:
        imu = pickle.load(open(row['pkl_path'], 'rb'))
        pos_rad = np.deg2rad(build_model_input_from_pkl(imu))
        id_df = parse_opensim_table(row['id_path'])
        ik_df = parse_opensim_table(row['mocap_ik_path'])
        t_id = id_df.index.to_numpy(dtype=np.float64)
        fs_hz = 1.0 / float(np.median(np.diff(t_id))) if len(t_id) > 2 else DEFAULT_FS_HZ
        mocap_pos = build_mocap_ik_rad(ik_df)

        imu_align = lpf_mc(pos_rad, fs_hz, ANGLE_CUTOFF, FILTER_ORDER, IN_MODE)
        mocap_align = lpf_mc(mocap_pos, fs_hz, ANGLE_CUTOFF, FILTER_ORDER, IN_MODE)
        lag, peak_info = estimate_lag_samples(
            imu_align, mocap_align, fs_hz, int(round(IK_ALIGN_MAX_LAG_SEC * fs_hz)), PEAK_THRESHOLD_DEG
        )

        start_imu, start_ref = max(lag, 0), max(-lag, 0)
        n_sync = min(len(pos_rad) - start_imu, len(mocap_pos) - start_ref, len(t_id) - start_ref)
        if n_sync < WINDOW_SIZE:
            raise RuntimeError(f'Synced window too short: {n_sync}')

        pos_sync = pos_rad[start_imu:start_imu + n_sync]
        t_sync = t_id[start_ref:start_ref + n_sync]
        id_nm = np.column_stack([id_df[c].to_numpy(dtype=np.float64) if c in id_df.columns else np.full(len(t_id), np.nan) for c in ID_COLS])
        id_nm_sync = id_nm[start_ref:start_ref + n_sync]

        pos_f = lpf_mc(pos_sync, fs_hz, ANGLE_CUTOFF, FILTER_ORDER, IN_MODE)
        vel_f = lpf_mc(np.gradient(pos_f, 1.0 / fs_hz, axis=0), fs_hz, VEL_CUTOFF, FILTER_ORDER, IN_MODE)
        pred_nmpkg = run_bilateral_inference(pos_f, vel_f)
        pred_nmpkg_f = lpf_mc(pred_nmpkg, fs_hz, OUT_CUTOFF, FILTER_ORDER, OUT_MODE)
        id_nmpkg_f = lpf_mc(id_nm_sync / row['mass_kg'], fs_hz, OUT_CUTOFF, FILTER_ORDER, OUT_MODE)

        metrics = []
        for c in range(6):
            rmse, r2 = rmse_r2(pred_nmpkg_f[:, c], id_nmpkg_f[:, c])
            metrics.append({'channel': CHANNELS[c], 'rmse_nmpkg': rmse, 'r2_nmpkg': r2})

        TRIAL_DATA[trial_key] = {
            'subject': subject, 'condition': cond, 't': t_sync, 'mass_kg': row['mass_kg'],
            'pred_nmpkg': pred_nmpkg_f, 'id_nmpkg': id_nmpkg_f, 'metrics': metrics,
            'lag_samples': lag, 'lag_seconds': lag / fs_hz, **peak_info,
        }
        rows.append({'trial': trial_key, 'n': n_sync, 'mean_rmse': np.nanmean([m['rmse_nmpkg'] for m in metrics]), 'mean_r2': np.nanmean([m['r2_nmpkg'] for m in metrics])})
        print(f'OK  {trial_key} | lag={lag:+d} samples | mean RMSE={rows[-1]["mean_rmse"]:.4f} R²={rows[-1]["mean_r2"]:.4f}')
    except Exception as exc:
        WARNINGS.append(f'[WARN] Failed {trial_key}: {exc}')
        print(f'FAIL {trial_key}: {exc}')

summary_df = pd.DataFrame(rows)
detail_rows = []
for trial, d in TRIAL_DATA.items():
    for m in d['metrics']:
        detail_rows.append({'trial': trial, **m})
detail_df = pd.DataFrame(detail_rows)

print(f'\nLoaded {len(TRIAL_DATA)} trials')
if not detail_df.empty:
    print('\nPer-joint mean across trials:')
    display(detail_df.groupby('channel')[['rmse_nmpkg', 'r2_nmpkg']].mean())
    print('\nOverall (all trials × joints):')
    print(f"  RMSE = {detail_df['rmse_nmpkg'].mean():.4f} N·m/kg")
    print(f"  R²   = {detail_df['r2_nmpkg'].mean():.4f}")

OK  AB02_Oscar::LG_0p8mps | lag=+1242 samples | mean RMSE=0.3322 R²=0.4296
OK  AB02_Oscar::RA_0p8mps | lag=+302 samples | mean RMSE=0.3987 R²=0.4018
OK  AB02_Oscar::RD_0p8mps | lag=+759 samples | mean RMSE=0.3672 R²=0.6187
OK  AB02_Oscar::LG_1p2mps | lag=+290 samples | mean RMSE=0.3350 R²=0.5970
OK  AB02_Oscar::LG_1p6mps | lag=+261 samples | mean RMSE=0.3721 R²=0.6550
OK  AB03_Ilseung::LG_0p8mps | lag=+232 samples | mean RMSE=0.0941 R²=0.8366
OK  AB03_Ilseung::RA_0p8mps | lag=+331 samples | mean RMSE=0.1563 R²=0.8983
OK  AB03_Ilseung::RD_0p8mps | lag=+263 samples | mean RMSE=0.1263 R²=0.8710
OK  AB03_Ilseung::LG_1p2mps | lag=+268 samples | mean RMSE=0.1019 R²=0.9185
OK  AB03_Ilseung::LG_1p6mps | lag=+227 samples | mean RMSE=0.1269 R²=0.9178
OK  AB06_Jimin::LG_0p8mps | lag=+215 samples | mean RMSE=0.1295 R²=0.7805
OK  AB06_Jimin::RA_0p8mps | lag=+854 samples | mean RMSE=0.1702 R²=0.8233
OK  AB06_Jimin::RD_0p8mps | lag=+559 samples | mean RMSE=0.5582 R²=0.4295
OK  AB06_Jimin::LG_1p2mps |

,rmse_nmpkg,r2_nmpkg
channel,,
ankle_angle_l,0.206669,0.892691
ankle_angle_r,0.213700,0.887382
hip_flexion_l,0.303713,0.630538
hip_flexion_r,0.345808,0.576613
knee_angle_l,0.149280,0.740085
knee_angle_r,0.210670,0.631461



Overall (all trials × joints):
  RMSE = 0.2383 N·m/kg
  R²   = 0.7265


In [ ]:
if not TRIAL_DATA:
    raise RuntimeError('No trials loaded. Check warnings above.')

trial_dd = widgets.Dropdown(options=sorted(TRIAL_DATA), description='Trial:')
unit_dd = widgets.Dropdown(options=['N·m/kg', 'N·m'], value='N·m/kg', description='Unit:')
time_slider = widgets.FloatRangeSlider(description='Time (s):', continuous_update=False, layout=widgets.Layout(width='700px'))
out = widgets.Output()


def _set_slider(trial_key):
    t = TRIAL_DATA[trial_key]['t']
    t_rel = t - t[0]
    time_slider.min, time_slider.max = float(t_rel[0]), float(t_rel[-1])
    time_slider.step = max((time_slider.max - time_slider.min) / 500, 1e-3)
    time_slider.value = (time_slider.min, time_slider.max)


def _draw(trial_key, unit, t_window):
    d = TRIAL_DATA[trial_key]
    t = d['t'] - d['t'][0]
    scale = 1.0 if unit == 'N·m/kg' else d['mass_kg']
    ylab = unit
    y_pred, y_id = d['pred_nmpkg'] * scale, d['id_nmpkg'] * scale
    t0, t1 = t_window
    names = ['Hip R', 'Knee R', 'Ankle R', 'Hip L', 'Knee L', 'Ankle L']

    fig, axs = plt.subplots(3, 2, figsize=(14, 10), sharex=True)
    for c, ax in enumerate(axs.reshape(-1)):
        m = d['metrics'][c]
        ax.plot(t, y_id[:, c], label='OpenSim ID', color='#1e88e5', lw=1.8)
        ax.plot(t, y_pred[:, c], label='Model', color='#e53935', lw=1.4, ls='--')
        ax.set_title(f"{names[c]} | RMSE={m['rmse_nmpkg']*scale:.3f}, R²={m['r2_nmpkg']:.3f}")
        ax.set_xlim(t0, t1)
        ax.set_ylabel(ylab)
        ax.axhline(0, color='gray', lw=0.6, ls=':')
        if c == 0:
            ax.legend()
    axs[-1, 0].set_xlabel('Time (s)')
    axs[-1, 1].set_xlabel('Time (s)')
    fig.suptitle(f"{trial_key} | sync: first hip peak | lag={d['lag_samples']:+d} samples", fontsize=12)
    fig.tight_layout()
    with out:
        out.clear_output(wait=True)
        plt.show()


def _redraw(*_):
    _draw(trial_dd.value, unit_dd.value, time_slider.value)


def _on_trial(change):
    _set_slider(change['new'])
    _redraw()

trial_dd.observe(_on_trial, names='value')
unit_dd.observe(_redraw, names='value')
time_slider.observe(_redraw, names='value')
_set_slider(trial_dd.value)
display(widgets.VBox([widgets.HBox([trial_dd, unit_dd]), time_slider, out]))
_redraw()